# 02 · Pipeline comparison

Run **different restoration pipelines on the same input, with the same degradation, scored by the same metrics**, and compare them. This drives the repo-root [`orchestrator.py`](../orchestrator.py) under the hood (via `nb.run_pipeline`), so every run is fair and reproducible.

The four endpoints:

| step | what it does | resolution |
|---|---|---|
| `joint` | denoise **and** super-resolve in one model | LR→HR |
| `sr` | spatial super-resolution | LR→HR |
| `denoise` | Noise2Noise U-Net denoiser | preserves res |
| `interp` | **temporal interpolation (my part)** | changes T |

Pipelines we compare here: **`joint`** vs the **`denoise → sr` cascade** (the thesis comparison), an **`sr`-only** baseline, and **`interp`**.

> Each `run_pipeline` call shells out to the orchestrator (can take minutes per run on GPU). Use `truncate=` to keep demo runs fast.

In [ ]:
%load_ext autoreload
%autoreload 2
import nb_utils as nb

cfg = nb.Config(data_dir="/srv/fMRI-data", bold_file=None)
bold = cfg.resolve_bold()
RUNS = cfg.out_dir / "pipeline_runs"
TRUNC = 20    # frames per run for a quick demo; set 0 for the whole run
print("input:", bold.name)
print("runs ->", RUNS)

## Run the pipelines

Architecture A (`degrade_once=yes`): the input is degraded **once** and fed to each pipeline, so `joint` and `denoise→sr` see the *same* noisy low-res input — apples to apples.

In [ ]:
results = []

results.append(nb.run_pipeline(["joint"],            bold, RUNS / "joint",   truncate=TRUNC, seed=0))
results.append(nb.run_pipeline(["denoise", "sr"],  bold, RUNS / "cascade", truncate=TRUNC, seed=0))
results.append(nb.run_pipeline(["sr"],              bold, RUNS / "sr_only", truncate=TRUNC, seed=0))

for r in results:
    print(f"  {r['label']:<14} rc={r['returncode']}  metrics={'ok' if r['metrics'] else 'MISSING'}")

## Comparison table

`psnr_db` / `ssim` are vs the clean reference (only defined when the time axis is unchanged); `tsnr_ratio` = output tSNR / reference tSNR (temporal signal stability — higher is smoother, which a denoiser should improve).

In [ ]:
df = nb.metrics_table(results)
df

In [ ]:
nb.bar_compare(df, "psnr_db", save_path=cfg.out_dir / "cmp_psnr.png")
nb.bar_compare(df, "ssim",    save_path=cfg.out_dir / "cmp_ssim.png")
nb.bar_compare(df, "tsnr_ratio", save_path=cfg.out_dir / "cmp_tsnr.png");

## Visual: degraded input vs pipeline output vs reference

The orchestrator writes montage slides for each run (`<run>/slides/*.png`). Here are the `joint` model's.

In [ ]:
nb.show_slides(RUNS / "joint", max_n=2)

In [ ]:
nb.show_slides(RUNS / "cascade", max_n=2)

## My part in a pipeline — temporal interpolation

`interp` changes the number of frames, so per-frame PSNR/SSIM vs the original is undefined (synthetic frames have no aligned ground truth). The orchestrator instead reports a **leave-out** PSNR/L1 (predict each held-out interior frame from its neighbours) under `interp_leaveout`, plus tSNR.

In [ ]:
interp_res = nb.run_pipeline(["interp"], bold, RUNS / "interp",
                             truncate=TRUNC, seed=0, interp_mode="fill-gaps")
m = interp_res["metrics"] or {}
print("output timepoints:", m.get("output_timepoints"),
      "(reference:", m.get("reference_timepoints"), ")")
print("tsnr_ratio:", m.get("tsnr_ratio"))
m.get("interp_leaveout", "(no leave-out reported)")

## Experiment ideas (just change the `steps` / flags)

```python
# isolate SR from denoising — feed the cascade a clean LR (no noise):
nb.run_pipeline(['denoise','sr'], bold, RUNS/'cascade_clean', truncate=TRUNC, noise='off')

# black-box chain (each endpoint self-degrades) — contrast baseline:
nb.run_pipeline(['denoise','sr'], bold, RUNS/'bbox', truncate=TRUNC, degrade_once='no')

# denoise then temporally interpolate:
nb.run_pipeline(['denoise','interp'], bold, RUNS/'denoise_interp', truncate=TRUNC)

# a different SR backbone:
nb.run_pipeline(['sr'], bold, RUNS/'sr_srcnn', truncate=TRUNC, sr_model='srcnn3d_deep')
```

Re-run the **Comparison table** cell after adding runs to `results` to fold them into the chart.